# Задание

Необходимо написать функцию оценки экспериментов.
Есть данные о пользователях и их покупках за 5 недель. На шестой неделе будет проводиться ряд экспериментов, целью которых было увеличение средней выручки с клиента. Задача функции - определять внедрять изменение или нет.

Информация про эксперименты:
- эксперименты считаем независимыми друг от друга;
- эксперименты проводились на шестой неделе, с 36 по 42 день включительно.
- размеры групп порядка 100-120;
- при дизайне экспериментов предполагалось, что будет применяться двусторонний тест Стьюдента с уровнем значимости 0.05.
- группы для некоторых экспериментов могли выбираться не из всей популяции, а из некоторого подмножества. Например, новые клиенты, которые не совершали покупок ранее.

Всего нужно будет оценить 1000 экспериментов. Оценка будет вычисляться по количеству неверно оценённых экспериментов согласно таблице:

| Количество ошибок | Оценка |
|------|-----|
| <=115 | 10 |
| 116-130 | 9 |
| 131-145 | 8 |
| 146-160 | 7 |
| 161-175 | 6 |
| 176-190 | 5 |
| 191-205 | 4 |
| 206-220 | 3 |
| 221-235 | 2 |
| 236-250 | 1 |
| >250 | 0 |

Дан набор исторических данных `df_sales_history.csv`. Для самопроверки предоставлена подвыборка из 200 экспериментов `experiments_sample.json` и данные о покупках для пользователей этих экспериментов во время эксперимента `df_sales_exp_sample.csv`.

In [1]:
import json
from typing import List

import numpy as np
import pandas as pd
from scipy import stats

ALPHA = 0.05

In [2]:
df_sales_history = pd.read_csv("df_sales_history.csv")
df_sales_exp = pd.read_csv("df_sales_exp_sample.csv")

df_sales_history["channel"] = df_sales_history["channel"].map(
    df_sales_history.groupby("channel")["cost"].mean()
)
df_sales_exp["channel"] = df_sales_exp["channel"].map(
    df_sales_exp.groupby("channel")["cost"].mean()
)

df_sales_history.drop(columns=["day"], inplace=True)
df_sales_exp.drop(columns=["day"], inplace=True)

revenue_user_exp = df_sales_exp.groupby("user_id").sum()
revenue_user_exp.reset_index(inplace=True)

revenue_user_hist = df_sales_history.groupby("user_id").sum()
revenue_user_hist.reset_index(inplace=True)

df = revenue_user_exp.merge(
    revenue_user_hist, on="user_id", how="outer", suffixes=("_exp", "_hist")
).fillna(0)
df.set_index("user_id", inplace=True)

target_feat = "cost_exp"
covar_feats = ["cost_hist", "channel_hist", "channel_exp"]

thetas = (
    df[[target_feat] + covar_feats].cov().values[0, 1:]
    / df[covar_feats].var(ddof=1).values
)


def check_test(a: List[int], b: List[int]):
    """Проверяет гипотезу.

    :param a: список id пользователей контрольной группы
    :param b: список id пользователей экспериментальной группы
    :return: 1 - внедряем изменение, 0 - иначе.
    """

    control = df.reindex(a, fill_value=0)
    exp = df.reindex(b, fill_value=0)

    exp = exp[exp["cost_exp"] > 0]
    control = control[control["cost_exp"] > 0]

    control_cuped = control[target_feat] - control[covar_feats].values.dot(thetas)
    exp_cuped = exp[target_feat] - exp[covar_feats].values.dot(thetas)

    _, pvalue = stats.ttest_ind(control_cuped, exp_cuped, equal_var=False)

    return int(pvalue < ALPHA)

In [ ]:
# тесты
with open("experiments_sample.json") as f:
    tests = json.load(f)

errors = []
for has_positive_effect, a, b in tests:
    res = check_test(a, b)
    assert res in [0, 1]
    errors.append(int(res != has_positive_effect))
print(f"Количество ошибок: {np.sum(errors)}\nДоля ошибок: {np.mean(errors)}")

Количество ошибок: 17
Доля ошибок: 0.085
